Mapping behaviors and reliance categories

In [1]:
import duckdb
import seaborn
from pathlib import Path

dataset = Path("../data/outputs/clean_analytical_dataset.parquet")
parameters = {"dataset": str(dataset)}

with duckdb.connect() as connection:
    statement = """
        SELECT semester, topic, reliance_category, COUNT(*) as interaction_count
        FROM read_parquet($dataset)
        GROUP BY semester, topic, reliance_category
        ORDER BY semester, topic, reliance_category
                """
    df = connection.execute(statement, parameters).df()

df

,semester,topic,reliance_category,interaction_count
0,f24,a1,CAUTIOUS_USE,229
1,f24,a1,REFLECTIVE_USE,330
2,f24,a1,THOUGHTLESS_USE,305
3,f24,a2,CAUTIOUS_USE,196
4,f24,a2,REFLECTIVE_USE,214
5,f24,a2,THOUGHTLESS_USE,260
6,f24,a3,CAUTIOUS_USE,244
7,f24,a3,REFLECTIVE_USE,412
8,f24,a3,THOUGHTLESS_USE,542
9,f24,a4,CAUTIOUS_USE,307


Count relevant interactions of userIds

In [2]:
with duckdb.connect() as connection:
    df_chats = connection.execute(
        """
        SELECT 
            userId,
            COUNT(*) AS total_interactions,
            SUM(CASE WHEN reliance_category = 'THOUGHTLESS_USE' THEN 1 ELSE 0 END) AS thoughtless_count,
            SUM(CASE WHEN reliance_category = 'REFLECTIVE_USE' THEN 1 ELSE 0 END) AS reflective_count,
            SUM(CASE WHEN reliance_category = 'CAUTIOUS_USE' THEN 1 ELSE 0 END) AS cautious_count
        FROM read_parquet($dataset)
        GROUP BY userId
        """,
        parameters,
    ).df()

df_chats

,userId,total_interactions,thoughtless_count,reflective_count,cautious_count
0,01cbf550-9041-70f1-3bbe-92176ffa992e,107,48.0,38.0,21.0
1,017b8560-6081-70e4-7303-6882d8635df0,16,7.0,4.0,5.0
2,113bc5a0-00c1-70a8-1558-53c8f78b1a99,36,13.0,14.0,9.0
3,b14bd5f0-0011-704d-7e86-b55f1b5f014a,14,4.0,7.0,3.0
4,e11ba5c0-60e1-7014-3010-a78f186a4362,33,10.0,14.0,9.0
...,...,...,...,...,...
198,e14bd5c0-2091-70a1-3e0d-252a10bc1372,22,8.0,10.0,4.0
199,d17b6520-5071-7048-0b67-edd7ac9f9e44,67,20.0,30.0,17.0
200,71cb5540-20c1-70a9-1806-fbc6af5f0b86,130,67.0,43.0,20.0
201,913b9550-d0d1-70fe-1817-55e26d83e2ef,5,5.0,0.0,0.0


Extract and explore grades data.

In [3]:
with duckdb.connect() as connection:
    df_grades = connection.execute(
        """
        SELECT *, 'f24' AS semester FROM read_csv($fall_sem)
        UNION ALL
        SELECT *, 's25' AS semester FROM read_csv($spring_sem)
        """,
        {
            "fall_sem": str(
                Path("/workspaces/StudyChat/scores/f24_grades_released_normalized.csv")
            ),
            "spring_sem": str(
                Path("/workspaces/StudyChat/scores/s25_grades_released_normalized.csv")
            ),
        },
    ).df()

df_grades

,userId,directory_name,a1,a2,a3,a4,a5,a6,a7,e1,e2,e3,llm,no_llm,semester
0,d1db45a0-f051-702b-f627-c33637d4528c,user_d1db45a0-f051-702b-f627-c33637d4528c,1.0,0.99,0.618182,0.967213,1.000000,1.000000,0.960396,0.9500,1.005,0.920,0.946667,0.958333,f24
1,c18b0590-1021-70b2-a04d-7da82bf20079,user_c18b0590-1021-70b2-a04d-7da82bf20079,1.0,0.88,0.781818,0.934426,0.907692,0.876106,1.000000,0.6200,0.940,0.600,0.908571,0.720000,f24
2,613b1510-4051-7027-1ee2-8f2c72d3c64b,user_613b1510-4051-7027-1ee2-8f2c72d3c64b,0.9,0.94,0.854545,0.934426,0.953846,1.000000,0.950495,0.8750,0.875,0.800,0.944762,0.850000,f24
3,315b9550-4081-7033-4f93-fe2be67cdfcc,user_315b9550-4081-7033-4f93-fe2be67cdfcc,1.0,0.95,0.872727,0.934426,1.000000,0.982301,1.000000,0.7350,0.790,0.725,0.965714,0.750000,f24
4,619bf560-4021-7015-9155-f148328587b8,user_619bf560-4021-7015-9155-f148328587b8,1.0,1.00,1.000000,0.967213,0.984615,0.982301,0.970297,0.9500,1.100,0.950,0.984762,1.000000,f24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,411bd580-f051-7091-c64b-39f944fda7b6,user_411bd580-f051-7091-c64b-39f944fda7b6,1.0,0.98,0.800000,1.000000,1.000000,0.902655,0.992593,0.9400,0.910,0.800,0.960254,0.883333,s25
177,61bbc5f0-2041-7061-260b-a4d10ca29e45,user_61bbc5f0-2041-7061-260b-a4d10ca29e45,1.0,1.00,0.963636,0.988889,1.000000,0.946903,1.000000,0.9300,1.050,0.910,0.985692,0.963333,s25
178,617b2500-4071-70e9-29f1-b4c4ce63bfaa,user_617b2500-4071-70e9-29f1-b4c4ce63bfaa,1.0,1.00,0.890909,0.977778,1.000000,0.973451,0.992593,0.8650,0.870,1.000,0.980922,0.911667,s25
179,d17b0530-40c1-7051-3339-0d2a5bb89823,user_d17b0530-40c1-7051-3339-0d2a5bb89823,1.0,0.94,0.963636,0.811111,1.000000,0.902655,0.933333,0.8700,1.010,0.920,0.928458,0.933333,s25


Identify the missing values.

In [4]:
chat_users = set(df_chats['userId'].unique())
grade_users = set(df_grades['userId'].unique())

missing_grades = chat_users - grade_users 
missing_chats = grade_users - chat_users 

print(f"Total students in chat logs: {len(chat_users)}")
print(f"Total students in grade files: {len(grade_users)}")
print(f"Students with chats but NO grades: {len(missing_grades)}")
print(f"Students with grades but NO chats: {len(missing_chats)}")

df_missing_28 = df_chats[df_chats['userId'].isin(missing_grades)]
df_missing_6 = df_grades[df_grades['userId'].isin(missing_chats)]

print("\nInteraction counts for the missing 28 students:")
print(df_missing_28['userId'].value_counts())
print("\n6 Students with no interactions:")
print(df_missing_6['userId'].value_counts())

Total students in chat logs: 203
Total students in grade files: 181
Students with chats but NO grades: 28
Students with grades but NO chats: 6

Interaction counts for the missing 28 students:
userId
710bd530-f031-7016-ebe5-ee614a366f14    1
912bd5e0-5061-7098-cb02-8d6eae25468c    1
e11ba5c0-60e1-7014-3010-a78f186a4362    1
c1abe560-20f1-70dc-393f-5a083011e23f    1
41fb4560-10f1-7016-b8cc-cd2bc89239b7    1
119b55f0-6001-70b7-9cf8-aff7a0dc1623    1
d14be530-3021-7083-6ee6-f96210849e13    1
f14b7530-60e1-704a-f8d5-4855523e6d94    1
81fbe5c0-8001-7054-3ac3-3e9db3f6e198    1
11cb0520-80d1-70f1-831e-2d3d54b32de3    1
210bd5c0-70d1-7070-b076-8b8ae197fb56    1
b1cb6570-80d1-70d6-b914-60bf3a31e5c7    1
d1cba5c0-4041-7099-95fd-aac383e0a3cb    1
819bd5f0-c0b1-70e2-e723-7bfd5aa9f3ca    1
d17b6520-5071-7048-0b67-edd7ac9f9e44    1
012b8580-9011-70e5-0aa0-480c3bb39caa    1
c1eb8590-5071-70c5-90a9-bcbf8ca28ccb    1
410bf580-d041-70ad-e668-e60717bb3b98    1
a1cb4570-5011-70b5-5fdf-1eaee3de61f4    1
91e

Investigate interactionCount = 0 for most prevalent reliance label.

In [5]:
with duckdb.connect() as connection:
    df_interaction_count_0 = connection.execute(
        """
        SELECT *
        FROM read_parquet($dataset)
        WHERE interactionCount = 0
        """,
        parameters,
    ).df()

df_interaction_count_0

,userId,chatId,semester,topic,interactionCount,llm_label,broad_label,specific_label,reliance_category,reliance_reason
0,613b1510-4051-7027-1ee2-8f2c72d3c64b,7363dd3f-199d-44c4-975b-52b4117fdf21,f24,a1,0,conceptual_questions>Programming Language,conceptual_questions,Programming Language,THOUGHTLESS_USE,"The student asks a very brief, context-free ho..."
1,613b1510-4051-7027-1ee2-8f2c72d3c64b,71e5ccea-04c1-4fa1-a1dd-ed847efe470d,f24,a1,0,conceptual_questions>Computer Science,conceptual_questions,Computer Science,CAUTIOUS_USE,The student is asking for an explanation of ho...
2,613b1510-4051-7027-1ee2-8f2c72d3c64b,56e24353-92d0-41fd-908b-ce427719909b,f24,a1,0,conceptual_questions>Programming Language,conceptual_questions,Programming Language,THOUGHTLESS_USE,"The student asks a very brief, context-free ho..."
3,016b65a0-90b1-703c-a8e3-caf440f1860a,d9dcc393-89e2-45be-a920-0b71de46a06f,f24,a1,0,off_topic>Greeting,off_topic,Greeting,THOUGHTLESS_USE,The student message is just a one-word greetin...
4,016b65a0-90b1-703c-a8e3-caf440f1860a,fe9bcaf8-c7d1-4d59-b08f-9ffc4cc401d3,f24,a1,0,off_topic>Greeting,off_topic,Greeting,THOUGHTLESS_USE,The student message is just a brief greeting w...
...,...,...,...,...,...,...,...,...,...,...
2191,e1cb9560-4091-706e-840f-87a913394f9e,94bbde99-2e79-4cab-817b-734ae502856d,s25,a7,0,conceptual_questions>Computer Science,conceptual_questions,Computer Science,CAUTIOUS_USE,The student is asking a conceptual question ab...
2192,e1cb9560-4091-706e-840f-87a913394f9e,8602d25a-2c72-461b-90b6-018d53de119f,s25,a7,0,editing_request>Edit English,editing_request,Edit English,REFLECTIVE_USE,"The student is iterating on experiments, expla..."
2193,c11bd560-50c1-70ac-8129-15966669f4ae,1f1c6743-4411-4df4-8ea1-48dd0b112f39,s25,a7,0,provide_context>Assignment Information,provide_context,Assignment Information,THOUGHTLESS_USE,The student pasted the assignment instructions...
2194,61abe5e0-d021-7066-78e5-8732da2c09a0,7fc7fba6-c7ba-4a7c-935c-01ab0dce7622,s25,a7,0,contextual_questions>Code Explanation,contextual_questions,Code Explanation,REFLECTIVE_USE,The student is working on a specific incomplet...


Thoughtless count jumps up to 56% when only accounting for the first interaction, whereas 40% when counting the full dataset.

In [6]:
with duckdb.connect() as connection:
    df_interaction_counted = connection.execute(
        """
        SELECT *
        FROM read_parquet($dataset)
        WHERE interactionCount != 0
        """,
        parameters,
    ).df()

df_interaction_counted

,userId,chatId,semester,topic,interactionCount,llm_label,broad_label,specific_label,reliance_category,reliance_reason
0,613b1510-4051-7027-1ee2-8f2c72d3c64b,71e5ccea-04c1-4fa1-a1dd-ed847efe470d,f24,a1,1,writing_request>Write Code,writing_request,Write Code,THOUGHTLESS_USE,"The student gives a very terse, context-free r..."
1,613b1510-4051-7027-1ee2-8f2c72d3c64b,71e5ccea-04c1-4fa1-a1dd-ed847efe470d,f24,a1,2,conceptual_questions>Programming Language,conceptual_questions,Programming Language,THOUGHTLESS_USE,The student asks for a direct coding solution ...
2,613b1510-4051-7027-1ee2-8f2c72d3c64b,71e5ccea-04c1-4fa1-a1dd-ed847efe470d,f24,a1,3,conceptual_questions>Programming Language,conceptual_questions,Programming Language,THOUGHTLESS_USE,"The student gives a very terse, context-free p..."
3,613b1510-4051-7027-1ee2-8f2c72d3c64b,56e24353-92d0-41fd-908b-ce427719909b,f24,a1,1,conceptual_questions>Programming Language,conceptual_questions,Programming Language,THOUGHTLESS_USE,The student gives only a terse follow-up phras...
4,613b1510-4051-7027-1ee2-8f2c72d3c64b,56e24353-92d0-41fd-908b-ce427719909b,f24,a1,2,conceptual_questions>Programming Language,conceptual_questions,Programming Language,THOUGHTLESS_USE,"The student gives a very terse, context-free p..."
...,...,...,...,...,...,...,...,...,...,...
14650,01ebb520-b0d1-709a-ca45-bb5b571f108f,0ef07679-1c4b-496e-8ba1-00b39999a5d8,s25,a7,1,provide_context>Error Message,provide_context,Error Message,CAUTIOUS_USE,The student is not just asking for a solution;...
14651,01ebb520-b0d1-709a-ca45-bb5b571f108f,0ef07679-1c4b-496e-8ba1-00b39999a5d8,s25,a7,2,provide_context>Other,provide_context,Other,REFLECTIVE_USE,The student is iterating on the prior debuggin...
14652,01ebb520-b0d1-709a-ca45-bb5b571f108f,0ef07679-1c4b-496e-8ba1-00b39999a5d8,s25,a7,3,provide_context>Error Message,provide_context,Error Message,REFLECTIVE_USE,The student is iterating on the code by runnin...
14653,01ebb520-b0d1-709a-ca45-bb5b571f108f,0ef07679-1c4b-496e-8ba1-00b39999a5d8,s25,a7,4,provide_context>Error Message,provide_context,Error Message,REFLECTIVE_USE,The student is iteratively modifying the code ...
